# Experimental ID-LoRA + LTX-2.5 — Kaggle Dual T4

**Compatibility warning:** ID-LoRA weights available publicly are trained for LTX-2.3. Their use on LTX-2.5 is experimental and not guaranteed. This notebook installs current ComfyUI native support (`LTXVReferenceAudio`), Manager, and downloads the LTX-2.5 low-VRAM model family plus an ID-LoRA adapter. It does not claim a certified two-stage ID-LoRA LTX-2.5 graph. Test at low resolution and inspect node compatibility first.

Enable 2× T4 in Kaggle. HF token may be required for gated Gemma files.

In [ ]:
# CELL 1 — Install ComfyUI, Manager, and current LTX support
import os, sys, subprocess, pathlib, torch
ROOT=pathlib.Path('/kaggle/working'); COMFY=ROOT/'ComfyUI'
def run(cmd,cwd=None): subprocess.run(cmd,cwd=cwd,check=True)
run(['nvidia-smi'])
assert torch.cuda.is_available(), 'Enable Kaggle GPU accelerator first.'
print('CUDA devices:',torch.cuda.device_count(),[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
if not COMFY.exists(): run(['git','clone','https://github.com/comfyanonymous/ComfyUI.git',str(COMFY)])
run([sys.executable,'-m','pip','install','-r',str(COMFY/'requirements.txt')])
CUSTOM=COMFY/'custom_nodes'; CUSTOM.mkdir(exist_ok=True)
manager=CUSTOM/'ComfyUI-Manager'
if not manager.exists(): run(['git','clone','https://github.com/Comfy-Org/ComfyUI-Manager.git',str(manager)])
# Native ID-LoRA support is in ComfyUI core (LTXVReferenceAudio); no legacy ID-LoRA custom node.
print('ComfyUI revision:'); run(['git','rev-parse','--short','HEAD'],cwd=COMFY)
print('Install complete. Native LTXVReferenceAudio availability will be checked in Cell 3 logs.')


In [ ]:
# CELL 2 — Download model files and place them in ComfyUI model folders
import os, pathlib, shutil
from huggingface_hub import hf_hub_download
ROOT=pathlib.Path('/kaggle/working'); COMFY=ROOT/'ComfyUI'; M=COMFY/'models'
TOKEN=os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACEHUB_API_TOKEN')
def get(repo,filename,folder,local_name=None):
    d=M/folder; d.mkdir(parents=True,exist_ok=True)
    target=d/(local_name or pathlib.Path(filename).name)
    if target.exists() and target.stat().st_size>1000000: print('Already exists',target); return target
    p=hf_hub_download(repo_id=repo,filename=filename,token=TOKEN)
    shutil.copy2(p,target); print('Downloaded',target,target.stat().st_size); return target
# LTX-2.5 lower-VRAM INT8 ConvRot model family. Verify current repo filenames if upstream revises them.
get('Lightricks/LTX-2.5','ltx-2.5-22b-dev-transformer-comfy-int8-convrot.safetensors','diffusion_models')
get('Lightricks/LTX-2.5','gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors','text_encoders')
get('Lightricks/LTX-2.5','ltx-2.5-video-vae-conv-bf16.safetensors','vae')
get('Lightricks/LTX-2.5','ltx-2.5-audio-vae-bf16.safetensors','vae')
get('Lightricks/LTX-2.5','ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors','latent_upscale_models')
get('Lightricks/LTX-2.5','ltx-2.5-22b-distilled-lora-450-bf16.safetensors','loras')
# ID-LoRA adapter is an LTX-2.3-trained LoRA; compatibility with LTX-2.5 is experimental.
get('AviadDahan/LTX-2.3-ID-LoRA-TalkVid-3K','lora_weights.safetensors','loras','id-lora-talkvid-ltx2.3.safetensors')
print('Model download stage complete. Gemma access may require accepting its Hugging Face terms and setting HF_TOKEN.')


In [ ]:
# CELL 3 — Launch ComfyUI with both T4s visible and conservative VRAM reservation
import os,sys,pathlib,subprocess,time,urllib.request
COMFY=pathlib.Path('/kaggle/working/ComfyUI')
os.environ['CUDA_VISIBLE_DEVICES']='0,1'
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
os.environ['PYTHONUNBUFFERED']='1'
cmd=[sys.executable,'main.py','--listen','0.0.0.0','--port','8188','--lowvram','--reserve-vram','2.0']
log=open('/kaggle/working/comfyui.log','w')
proc=subprocess.Popen(cmd,cwd=COMFY,stdout=log,stderr=subprocess.STDOUT)
for n in range(90):
    time.sleep(2)
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats',timeout=3); print('ComfyUI ready on port 8188'); break
    except Exception:
        if proc.poll() is not None: raise RuntimeError(pathlib.Path('/kaggle/working/comfyui.log').read_text(errors='ignore')[-10000:])
else: print('Still starting; inspect /kaggle/working/comfyui.log')
print('Both GPUs visible:',os.environ['CUDA_VISIBLE_DEVICES']); print('Important: CUDA_VISIBLE_DEVICES does not pool VRAM. Native ComfyUI may not shard a single transformer over both T4s.')
